# PeMS 检测站道路路段匹配与可视化

本 notebook 实现：
1. 加载 Caltrans 官方路网数据（SHN_Lines）
2. 根据里程桩（Odometer）匹配 PeMS 检测站对应的道路路段
3. 在地图上可视化检测站及其覆盖的道路路段
4. 校正检测站坐标偏差

## 1. 环境配置

In [ ]:
# 安装必要的库
!pip install geopandas shapely folium pandas numpy -q

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import LineString, Point, MultiLineString
from shapely.ops import substring, linemerge
import folium
from folium.plugins import MarkerCluster
import os
import warnings
warnings.filterwarnings('ignore')

print("库加载完成！")

## 2. 配置数据路径

**请根据实际情况修改以下路径**

In [ ]:
# ============== 配置区域 ==============

# Caltrans 路网数据路径
SHN_LINES_PATH = "SHN_Lines/SHN_Lines.shp"  # 道路中心线
SHN_POSTMILES_PATH = "SHN_Postmiles_Tenth/SHN_Postmiles_Tenth.shp"  # 里程桩点位（可选）

# PeMS 元数据路径
PEMS_META_PATH = "d03_text_meta_2025_01_18.txt"  # 修改为你的文件

# 输出目录
OUTPUT_DIR = "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("配置完成！")

## 3. 加载数据

In [ ]:
# 加载 Caltrans 路网数据
print("正在加载 Caltrans 路网数据...")
lines_raw = gpd.read_file(SHN_LINES_PATH)
print(f"原始坐标系: {lines_raw.crs}")

# 转换坐标系到 WGS84 (EPSG:4326) 用于 Folium 显示
lines = lines_raw.to_crs(epsg=4326)
print(f"转换后坐标系: {lines.crs}")
print(f"Caltrans 路段数: {len(lines)}")

# 查看字段
print(f"\n字段列表: {lines.columns.tolist()}")

In [ ]:
# 加载 PeMS 元数据
print("正在加载 PeMS 元数据...")

# 定义列名
pems_columns = [
    'ID', 'Fwy', 'Dir', 'District', 'County', 'City',
    'State_PM', 'Abs_PM', 'Latitude', 'Longitude', 'Length',
    'Type', 'Lanes', 'Name', 'User_ID_1', 'User_ID_2',
    'User_ID_3', 'User_ID_4'
]

pems = pd.read_csv(
    PEMS_META_PATH,
    sep='\t',
    names=pems_columns,
    header=0,
    dtype={'ID': str, 'Fwy': str, 'Dir': str, 'Type': str}
)

print(f"PeMS 检测站数: {len(pems)}")
print(f"\n数据预览:")
display(pems[['ID', 'Fwy', 'Dir', 'Type', 'Abs_PM', 'Length', 'Name']].head(10))

In [ ]:
# 查看 PeMS 数据中的高速公路和方向
print("PeMS 数据统计:")
print(f"\n高速公路: {sorted(pems['Fwy'].unique())}")
print(f"方向: {pems['Dir'].unique()}")
print(f"类型: {pems['Type'].unique()}")

In [ ]:
# 查看 Caltrans 数据中的路线和方向
print("Caltrans 数据统计:")
print(f"\n路线示例: {sorted(lines['Route'].unique())[:20]}")
print(f"方向: {lines['Direction'].unique()}")

## 4. 定义匹配函数

In [ ]:
def convert_direction(pems_dir):
    """
    PeMS 方向转换为 Caltrans 方向
    PeMS: E, W, N, S
    Caltrans: EB, WB, NB, SB
    """
    mapping = {'E': 'EB', 'W': 'WB', 'N': 'NB', 'S': 'SB'}
    return mapping.get(pems_dir, pems_dir)


def find_road_segments(station, lines_gdf):
    """
    为检测站找到对应的道路路段
    
    Parameters:
    - station: PeMS 检测站行（Series）
    - lines_gdf: Caltrans 路网 GeoDataFrame
    
    Returns:
    - 匹配的路段几何（LineString）或 None
    - 校正后的坐标 (lat, lon) 或 None
    """
    route = str(station['Fwy'])
    direction = convert_direction(station['Dir'])
    abs_pm = station['Abs_PM']
    length = station['Length'] if pd.notna(station['Length']) and station['Length'] > 0 else 0.5
    
    if pd.isna(abs_pm):
        return None, None
    
    # 计算检测站覆盖范围
    pm_start = abs_pm
    pm_end = abs_pm + length
    
    # 筛选匹配的路线和方向
    candidates = lines_gdf[
        (lines_gdf['Route'].astype(str) == route) &
        (lines_gdf['Direction'] == direction)
    ].copy()
    
    if len(candidates) == 0:
        # 尝试不限方向
        candidates = lines_gdf[lines_gdf['Route'].astype(str) == route].copy()
        if len(candidates) == 0:
            return None, None
    
    # 查找包含该里程范围的路段
    matched_segments = []
    for _, seg in candidates.iterrows():
        seg_start = seg['bOdometer']
        seg_end = seg['eOdometer']
        
        # 处理里程桩递减的情况
        if seg_start > seg_end:
            seg_start, seg_end = seg_end, seg_start
        
        # 检查是否有重叠
        if seg_start <= pm_end and seg_end >= pm_start:
            matched_segments.append(seg)
    
    if not matched_segments:
        return None, None
    
    # 截取路段并计算校正坐标
    return extract_and_correct(matched_segments, pm_start, pm_end)


def extract_and_correct(segments, pm_start, pm_end):
    """
    从匹配的路段中截取检测站覆盖的部分，并计算校正坐标
    """
    all_coords = []
    
    for seg in segments:
        geom = seg.geometry
        seg_start = seg['bOdometer']
        seg_end = seg['eOdometer']
        
        # 处理里程桩递减
        reversed_pm = False
        if seg_start > seg_end:
            seg_start, seg_end = seg_end, seg_start
            reversed_pm = True
        
        seg_length = seg_end - seg_start
        if seg_length <= 0:
            continue
        
        # 计算截取比例
        ratio_start = max(0, (pm_start - seg_start) / seg_length)
        ratio_end = min(1, (pm_end - seg_start) / seg_length)
        
        if ratio_start >= ratio_end:
            continue
        
        # 如果里程桩是反向的，需要反转比例
        if reversed_pm:
            ratio_start, ratio_end = 1 - ratio_end, 1 - ratio_start
        
        try:
            # 使用 shapely 的 substring 截取
            sub_geom = substring(geom, ratio_start, ratio_end, normalized=True)
            if sub_geom and not sub_geom.is_empty:
                if sub_geom.geom_type == 'LineString':
                    all_coords.extend(list(sub_geom.coords))
                elif sub_geom.geom_type == 'Point':
                    all_coords.append((sub_geom.x, sub_geom.y))
        except Exception as e:
            continue
    
    if len(all_coords) >= 2:
        line = LineString(all_coords)
        # 校正坐标：取路段中点
        midpoint = line.interpolate(0.5, normalized=True)
        corrected_coords = (midpoint.y, midpoint.x)  # (lat, lon)
        return line, corrected_coords
    elif len(all_coords) == 1:
        point = Point(all_coords[0])
        corrected_coords = (point.y, point.x)
        return point, corrected_coords
    
    return None, None


print("匹配函数定义完成！")

## 5. 执行批量匹配

In [ ]:
def match_all_stations(pems_df, lines_gdf, verbose=True):
    """
    为所有检测站匹配道路路段
    """
    results = []
    matched = 0
    total = len(pems_df)
    
    for idx, station in pems_df.iterrows():
        road_geom, corrected_coords = find_road_segments(station, lines_gdf)
        
        result = {
            'ID': station['ID'],
            'Fwy': station['Fwy'],
            'Dir': station['Dir'],
            'Type': station['Type'],
            'Name': station['Name'],
            'Abs_PM': station['Abs_PM'],
            'Length': station['Length'],
            'Latitude': station['Latitude'],
            'Longitude': station['Longitude'],
            'road_geometry': road_geom,
            'matched': road_geom is not None
        }
        
        # 添加校正坐标
        if corrected_coords:
            result['corrected_lat'] = corrected_coords[0]
            result['corrected_lon'] = corrected_coords[1]
        else:
            result['corrected_lat'] = station['Latitude']
            result['corrected_lon'] = station['Longitude']
        
        results.append(result)
        
        if road_geom is not None:
            matched += 1
        
        if verbose and (idx + 1) % 200 == 0:
            print(f"已处理 {idx + 1}/{total}，匹配成功 {matched} ({matched/(idx+1)*100:.1f}%)")
    
    print(f"\n匹配完成: {matched}/{total} ({matched/total*100:.1f}%)")
    return pd.DataFrame(results)


# 执行匹配
print("开始匹配检测站与道路路段...")
print("这可能需要几分钟时间...\n")
matched_df = match_all_stations(pems, lines)

In [ ]:
# 查看匹配结果统计
print("匹配结果统计:")
print(f"总站点数: {len(matched_df)}")
print(f"匹配成功: {matched_df['matched'].sum()}")
print(f"匹配失败: {(~matched_df['matched']).sum()}")

print("\n按类型统计匹配情况:")
match_by_type = matched_df.groupby('Type')['matched'].agg(['sum', 'count'])
match_by_type['ratio'] = (match_by_type['sum'] / match_by_type['count'] * 100).round(1)
match_by_type.columns = ['匹配成功', '总数', '匹配率(%)']
display(match_by_type)

In [ ]:
# 查看未匹配的站点
unmatched = matched_df[~matched_df['matched']]
print(f"未匹配站点数: {len(unmatched)}")
if len(unmatched) > 0:
    print("\n未匹配站点示例:")
    display(unmatched[['ID', 'Fwy', 'Dir', 'Type', 'Abs_PM', 'Name']].head(20))

## 6. 坐标校正效果分析

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """计算两点间的球面距离（米）"""
    R = 6371000  # 地球半径（米）
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c


# 计算校正距离
matched_only = matched_df[matched_df['matched']].copy()
matched_only['correction_distance'] = matched_only.apply(
    lambda row: haversine_distance(
        row['Latitude'], row['Longitude'],
        row['corrected_lat'], row['corrected_lon']
    ) if pd.notna(row['Latitude']) else np.nan,
    axis=1
)

print("坐标校正距离统计（米）:")
print(matched_only['correction_distance'].describe())

print("\n校正距离分布:")
bins = [0, 10, 50, 100, 200, 500, 1000, float('inf')]
labels = ['0-10m', '10-50m', '50-100m', '100-200m', '200-500m', '500-1000m', '>1000m']
matched_only['distance_bin'] = pd.cut(matched_only['correction_distance'], bins=bins, labels=labels)
print(matched_only['distance_bin'].value_counts().sort_index())

## 7. 可视化地图

In [ ]:
def create_road_segment_map(matched_df, title="PeMS Stations with Road Segments",
                            show_original=True, show_corrected=True, show_segments=True):
    """
    创建带道路路段的地图
    
    Parameters:
    - matched_df: 匹配结果 DataFrame
    - title: 地图标题
    - show_original: 是否显示原始坐标点
    - show_corrected: 是否显示校正坐标点
    - show_segments: 是否显示道路路段
    """
    # 过滤有效数据
    valid_df = matched_df[
        matched_df['Latitude'].notna() & 
        matched_df['Longitude'].notna()
    ].copy()
    
    # 计算中心点
    center_lat = valid_df['Latitude'].mean()
    center_lon = valid_df['Longitude'].mean()
    
    # 创建地图
    m = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles='OpenStreetMap')
    
    # 站点类型颜色
    type_colors = {
        'ML': 'blue',
        'OR': 'green',
        'FR': 'red',
        'HV': 'purple',
        'FF': 'orange',
        'CD': 'darkblue',
        'CH': 'gray'
    }
    
    # 创建图层
    if show_segments:
        segment_group = folium.FeatureGroup(name='Road Segments (路段)')
    if show_original:
        original_group = folium.FeatureGroup(name='Original Coords (原始坐标)')
    if show_corrected:
        corrected_group = folium.FeatureGroup(name='Corrected Coords (校正坐标)')
    
    matched_count = 0
    
    for _, row in valid_df.iterrows():
        color = type_colors.get(row['Type'], 'gray')
        
        popup_html = f"""
        <b>Station ID:</b> {row['ID']}<br>
        <b>Name:</b> {row['Name']}<br>
        <b>Freeway:</b> {row['Fwy']} {row['Dir']}<br>
        <b>Type:</b> {row['Type']}<br>
        <b>Abs_PM:</b> {row['Abs_PM']}<br>
        <b>Length:</b> {row['Length']} mi<br>
        <b>Matched:</b> {'✓' if row['matched'] else '✗'}
        """
        
        # 添加道路路段
        if show_segments and row['road_geometry'] is not None:
            geom = row['road_geometry']
            if geom.geom_type == 'LineString':
                coords = [(lat, lon) for lon, lat in geom.coords]
                folium.PolyLine(
                    coords,
                    color=color,
                    weight=5,
                    opacity=0.8,
                    popup=folium.Popup(popup_html, max_width=300)
                ).add_to(segment_group)
                matched_count += 1
        
        # 添加原始坐标点
        if show_original:
            folium.CircleMarker(
                location=[row['Latitude'], row['Longitude']],
                radius=4,
                color='black',
                fill=True,
                fillColor=color,
                fillOpacity=0.7,
                popup=folium.Popup(popup_html + "<br><b>(Original)</b>", max_width=300)
            ).add_to(original_group)
        
        # 添加校正坐标点
        if show_corrected and row['matched'] and pd.notna(row['corrected_lat']):
            folium.CircleMarker(
                location=[row['corrected_lat'], row['corrected_lon']],
                radius=4,
                color=color,
                fill=True,
                fillColor=color,
                fillOpacity=1.0,
                popup=folium.Popup(popup_html + "<br><b>(Corrected)</b>", max_width=300)
            ).add_to(corrected_group)
    
    # 添加图层到地图
    if show_segments:
        segment_group.add_to(m)
    if show_original:
        original_group.add_to(m)
    if show_corrected:
        corrected_group.add_to(m)
    
    # 添加图层控制
    folium.LayerControl().add_to(m)
    
    # 添加图例
    legend_html = f'''
    <div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000;
                background: white; padding: 10px; border-radius: 5px;
                border: 2px solid gray; font-size: 12px;">
    <b>Station Types</b><br>
    <i style="background:blue; width:12px; height:12px; display:inline-block;"></i> ML - Mainline<br>
    <i style="background:green; width:12px; height:12px; display:inline-block;"></i> OR - On Ramp<br>
    <i style="background:red; width:12px; height:12px; display:inline-block;"></i> FR - Off Ramp<br>
    <i style="background:purple; width:12px; height:12px; display:inline-block;"></i> HV - HOV<br>
    <i style="background:orange; width:12px; height:12px; display:inline-block;"></i> FF - Fwy-Fwy<br>
    <hr>
    <b>Stats</b><br>
    Stations: {len(valid_df)}<br>
    Matched: {matched_count}
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    # 添加标题
    title_html = f'''
    <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 1000;
                background: white; padding: 10px; border-radius: 5px;
                border: 2px solid gray; font-size: 16px; font-weight: bold;">
    {title}
    </div>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    return m


print("可视化函数定义完成！")

In [ ]:
# 创建完整地图（显示所有元素）
print("正在生成地图...")
full_map = create_road_segment_map(
    matched_df,
    title="PeMS D3 Stations with Road Segments",
    show_original=True,
    show_corrected=True,
    show_segments=True
)

# 保存地图
map_path = os.path.join(OUTPUT_DIR, 'pems_road_segments_full.html')
full_map.save(map_path)
print(f"地图已保存: {map_path}")

# 显示地图
full_map

## 8. 针对特定高速公路的可视化

In [ ]:
def create_freeway_map(matched_df, freeway, direction=None, title=None):
    """
    为特定高速公路创建地图
    
    Parameters:
    - matched_df: 匹配结果 DataFrame
    - freeway: 高速公路编号（如 '80', '5'）
    - direction: 方向（'N', 'S', 'E', 'W'），None 表示双向
    """
    # 筛选数据
    df_fwy = matched_df[matched_df['Fwy'] == str(freeway)].copy()
    
    if direction:
        df_fwy = df_fwy[df_fwy['Dir'] == direction]
    
    if len(df_fwy) == 0:
        print(f"未找到高速公路 {freeway} {direction or ''} 的数据")
        return None
    
    if title is None:
        title = f"I-{freeway} {direction or 'Both Directions'} - Road Segments"
    
    print(f"筛选到 {len(df_fwy)} 个站点")
    
    return create_road_segment_map(df_fwy, title=title)


# 示例：可视化 I-80
map_80 = create_freeway_map(matched_df, freeway='80', direction=None)
if map_80:
    map_80.save(os.path.join(OUTPUT_DIR, 'i80_road_segments.html'))
    print(f"地图已保存: {OUTPUT_DIR}/i80_road_segments.html")
    display(map_80)

In [ ]:
# 可视化 I-5
map_5 = create_freeway_map(matched_df, freeway='5', direction=None)
if map_5:
    map_5.save(os.path.join(OUTPUT_DIR, 'i5_road_segments.html'))
    print(f"地图已保存: {OUTPUT_DIR}/i5_road_segments.html")
    display(map_5)

## 9. 导出校正后的数据

In [ ]:
# 导出校正后的坐标数据（不含几何对象）
export_df = matched_df.drop(columns=['road_geometry']).copy()

# 保存为 CSV
csv_path = os.path.join(OUTPUT_DIR, 'pems_stations_corrected.csv')
export_df.to_csv(csv_path, index=False)
print(f"校正数据已保存: {csv_path}")

# 预览
print("\n校正数据预览:")
display(export_df[['ID', 'Fwy', 'Dir', 'Type', 'Latitude', 'Longitude', 
                   'corrected_lat', 'corrected_lon', 'matched']].head(10))

In [ ]:
# 导出为 GeoJSON（包含路段几何）
def export_to_geojson(matched_df, output_path):
    """
    将匹配结果导出为 GeoJSON
    """
    # 只导出有几何数据的行
    valid_df = matched_df[matched_df['road_geometry'].notna()].copy()
    
    # 创建 GeoDataFrame
    gdf = gpd.GeoDataFrame(
        valid_df.drop(columns=['road_geometry']),
        geometry=valid_df['road_geometry'].tolist(),
        crs='EPSG:4326'
    )
    
    gdf.to_file(output_path, driver='GeoJSON')
    print(f"GeoJSON 已保存: {output_path}")
    return gdf


geojson_path = os.path.join(OUTPUT_DIR, 'pems_road_segments.geojson')
gdf_export = export_to_geojson(matched_df, geojson_path)

## 10. 使用说明

### 地图图层说明

| 图层 | 说明 |
|------|------|
| Road Segments | 检测站覆盖的道路路段（粗线条） |
| Original Coords | PeMS 原始坐标（黑边圆点） |
| Corrected Coords | 校正后的坐标（纯色圆点） |

### 颜色含义

| 颜色 | 类型 |
|------|------|
| 🔵 蓝色 | ML - 主线 |
| 🟢 绿色 | OR - 入口匝道 |
| 🔴 红色 | FR - 出口匝道 |
| 🟣 紫色 | HV - HOV车道 |
| 🟠 橙色 | FF - 高速互通 |

### 输出文件

| 文件 | 说明 |
|------|------|
| `pems_road_segments_full.html` | 完整地图 |
| `pems_stations_corrected.csv` | 校正后的坐标数据 |
| `pems_road_segments.geojson` | 路段几何数据 |